# LLM Classification Finetuning: Level 1 length + style stacking (llm-cls-stack)

**Settings:** Accelerator = **GPU T4 x2**, Internet = **OFF**

**Inputs (Add Input):** competition data + Your Work -> Notebooks -> **llm-cls-train**

**Idea:** model ovvoru response-ayum 200 tokens-la cut pannuthu, so response length-a paaka mudiyaathu. Aana Chatbot Arena-la length thaan strongest style signal.
1. Model paakaatha training rows-la (training-la use pannaatha ~35k rows) TTA predictions edukrom.
2. Model probabilities + length / markdown features vachu oru chinna second-stage model (stacker) train pannrom.
3. Val 2000-la base TTA vs Logistic Regression vs LightGBM compare. Best-a iruntha mattum use, illana TTA-ve.

In [ ]:
import os, glob, json, time, re, threading
import numpy as np, pandas as pd, torch
from sklearn.metrics import log_loss
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def find(pattern):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    assert hits, f"input-la kedaikala: {pattern} (Add Input pannina?)"
    return hits[0]

MODEL_DIR  = os.path.dirname(find("qwen-cls/config.json"))
VAL_PATH   = find("val.parquet")
TRAIN_PATH = find("train.csv")
TEST_PATH  = find("test.csv")
INFO       = json.load(open(find("train_info.json")))

RERUN = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
N_GPU = torch.cuda.device_count()
print("train info:", INFO)
print("RERUN:", RERUN, "| GPUs:", N_GPU)

In [ ]:
STACK_N = 10000                               # stacker train panna evvalavu unseen rows
SEED    = 42                                  # training notebook-la use panna same seed
N_VAL, N_TRAIN = INFO["n_val"], INFO["n_train"]
MAX_P, MAX_R   = INFO["max_p"], INFO["max_r"]

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
PAD_ID = tok.pad_token_id

def parse(x):
    try:
        v = json.loads(x)
        if isinstance(v, list):
            return "\n".join("" if s is None else str(s) for s in v)
        return str(v)
    except Exception:
        return str(x)

H_P = tok("Prompt: ", add_special_tokens=False)["input_ids"]
H_A = tok("\n\nResponse A: ", add_special_tokens=False)["input_ids"]
H_B = tok("\n\nResponse B: ", add_special_tokens=False)["input_ids"]
H_Q = tok("\n\nWhich response is better?", add_special_tokens=False)["input_ids"]

def build(d, swap=False):
    a_col, b_col = ("response_b", "response_a") if swap else ("response_a", "response_b")
    P = tok(d["prompt"].tolist(), add_special_tokens=False, truncation=True, max_length=MAX_P)["input_ids"]
    A = tok(d[a_col].tolist(),    add_special_tokens=False, truncation=True, max_length=MAX_R)["input_ids"]
    B = tok(d[b_col].tolist(),    add_special_tokens=False, truncation=True, max_length=MAX_R)["input_ids"]
    return [H_P + p + H_A + a + H_B + b + H_Q for p, a, b in zip(P, A, B)]

In [ ]:
from tqdm.auto import tqdm

def collate(seqs, device):
    L = max(len(s) for s in seqs)
    ids  = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
    mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)]  = torch.tensor(s)
        mask[i, :len(s)] = 1
    return {"input_ids": ids.to(device), "attention_mask": mask.to(device)}

@torch.inference_mode()
def predict(model, ids, bs=16, device="cuda:0", progress=True):
    order = np.argsort([-len(x) for x in ids], kind="stable")      # length-sorted batching
    out = np.zeros((len(ids), 3), dtype=np.float32)
    starts = range(0, len(ids), bs)
    if progress:
        starts = tqdm(starts, leave=False)
    with torch.cuda.device(device):
        for i in starts:
            idx = order[i:i + bs]
            batch = collate([ids[j] for j in idx], device)
            logits = model(**batch, use_cache=False).logits.float()
            out[idx] = torch.softmax(logits, -1).cpu().numpy()
    return out

def predict_2gpu(models, ids, bs=16):
    half = len(ids) // 2
    parts, res = [ids[:half], ids[half:]], [None, None]
    def run(k):
        res[k] = predict(models[k], parts[k], bs=bs, device=f"cuda:{k}", progress=False)
    threads = [threading.Thread(target=run, args=(k,)) for k in range(2)]
    for t in threads: t.start()
    for t in threads: t.join()
    return np.concatenate(res)

def load(device):
    m = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, dtype=torch.float16,
                                                           device_map={"": device})
    return m.eval()

def run_all(ids):
    return predict_2gpu(MODELS, ids) if len(MODELS) > 1 else predict(MODELS[0], ids)

def tta_probs(d):
    # swap-TTA: normal + A/B swap, swap-la A/B columns thirumba maathi average
    p1 = run_all(build(d))
    p2 = run_all(build(d, swap=True))[:, [1, 0, 2]]
    return (p1 + p2) / 2

## Data: training-oda same shuffle-a recreate panni, model paakaatha rows-a edukrom

In [ ]:
df = pd.read_csv(TRAIN_PATH)
for c in ["prompt", "response_a", "response_b"]:
    df[c] = df[c].apply(parse)
df["label"] = df[["winner_model_a", "winner_model_b", "winner_tie"]].values.argmax(1)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

val_df = pd.read_parquet(VAL_PATH)
# Safety check: shuffle correct-a recreate aagiducha? (illana training rows leak aagum)
assert (df["id"].values[:N_VAL] == val_df["id"].values).all(), "shuffle match aagala, STOP"
print("shuffle match ✅")

# Model train pannatha rows: v1 = rows 22000+, v2 = rows 2000-11999 (+ 42000+). train_info-la irunthu edukrom
tr_start, tr_end = map(int, str(INFO.get("train_rows", f"{N_VAL}-{N_VAL + N_TRAIN}")).split("-"))
unseen   = pd.concat([df.iloc[N_VAL:tr_start], df.iloc[tr_end:]]).reset_index(drop=True)
print("model trained on rows:", tr_start, "-", tr_end)
stack_df = unseen.iloc[:STACK_N].reset_index(drop=True)
print("unseen rows:", len(unseen), "| stack rows:", len(stack_df), "| val rows:", len(val_df))

test = pd.read_csv(TEST_PATH)
for c in ["prompt", "response_a", "response_b"]:
    test[c] = test[c].apply(parse)
print("test rows:", len(test))

In [ ]:
MODELS = [load("cuda:0")] + ([load("cuda:1")] if N_GPU > 1 else [])
print("models loaded on", len(MODELS), "GPU(s)")

In [ ]:
t = time.time(); p_val   = tta_probs(val_df);   print(f"val   TTA: {time.time()-t:.0f}s")
t = time.time(); p_stack = tta_probs(stack_df); print(f"stack TTA: {time.time()-t:.0f}s")
t = time.time(); p_test  = tta_probs(test);     print(f"test  TTA: {time.time()-t:.0f}s")

y_val, y_stack = val_df["label"].values, stack_df["label"].values
base_ll = log_loss(y_val, p_val, labels=[0, 1, 2])
print(f"BASE (TTA only) val log loss: {base_ll:.4f}")

## Features: length + markdown style (model-ku 200 tokens-ku mela theriyaathathu)

In [ ]:
LIST_RE = re.compile(r"^\s*([-*+]|\d+\.)\s")

def resp_stats(s):
    s = s or ""
    lines = s.split("\n")
    return dict(
        chars   = len(s),
        words   = len(s.split()),
        lines   = len(lines),
        headers = sum(l.lstrip().startswith("#") for l in lines),
        lists   = sum(bool(LIST_RE.match(l)) for l in lines),
        bold    = s.count("**") // 2,
        code    = s.count("```") // 2,
        empty   = int(len(s.strip()) == 0),
    )

STATS = {}
def stats_df(d, col):
    key = (id(d), col)
    if key not in STATS:
        STATS[key] = pd.DataFrame([resp_stats(x) for x in d[col]])
    return STATS[key]

def feats(d, probs, swap=False):
    a_col, b_col = ("response_b", "response_a") if swap else ("response_a", "response_b")
    A, B = stats_df(d, a_col), stats_df(d, b_col)
    p = probs[:, [1, 0, 2]] if swap else probs
    F = pd.DataFrame(index=range(len(d)))
    eps = 1e-6
    F["lp_a"], F["lp_b"], F["lp_tie"] = np.log(p[:, 0] + eps), np.log(p[:, 1] + eps), np.log(p[:, 2] + eps)
    for k in A.columns:
        if k == "empty":
            F["a_empty"], F["b_empty"] = A[k].values, B[k].values
        else:
            F[f"a_{k}"] = np.log1p(A[k].values)
            F[f"b_{k}"] = np.log1p(B[k].values)
            F[f"d_{k}"] = F[f"a_{k}"] - F[f"b_{k}"]
    F["same"] = (d[a_col].values == d[b_col].values).astype(int)
    F["prompt_len"] = np.log1p(d["prompt"].str.len().values)
    return F

SWAP_Y = np.array([1, 0, 2])

# Stacker training data: normal + swapped (position-symmetric stacker)
X_tr = pd.concat([feats(stack_df, p_stack), feats(stack_df, p_stack, swap=True)], ignore_index=True)
y_tr = np.concatenate([y_stack, SWAP_Y[y_stack]])
print("stacker train:", X_tr.shape, "| features:", list(X_tr.columns))

In [ ]:
def sym_predict(model, d, probs):
    # Stacker-kkum TTA: normal + swapped average
    p1 = model.predict_proba(feats(d, probs))
    p2 = model.predict_proba(feats(d, probs, swap=True))[:, [1, 0, 2]]
    return (p1 + p2) / 2

results = [dict(method="base TTA", val_log_loss=round(base_ll, 4))]
models = {}

# 1) Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
lr = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000))
lr.fit(X_tr, y_tr)
p_lr = sym_predict(lr, val_df, p_val)
results.append(dict(method="LR stack", val_log_loss=round(log_loss(y_val, p_lr, labels=[0, 1, 2]), 4)))
models["LR stack"] = lr

# 2) LightGBM (early stopping stack data-oda 10%-la, val-a thodaama)
try:
    import lightgbm as lgb
    n = len(stack_df); cut = int(n * 0.9)
    tr_idx = np.r_[0:cut, n:n + cut]; es_idx = np.r_[cut:n, n + cut:2 * n]
    gbm = lgb.LGBMClassifier(objective="multiclass", n_estimators=3000, learning_rate=0.02,
                             num_leaves=15, min_child_samples=50, subsample=0.8, subsample_freq=1,
                             colsample_bytree=0.8, random_state=SEED, verbose=-1)
    gbm.fit(X_tr.iloc[tr_idx], y_tr[tr_idx],
            eval_set=[(X_tr.iloc[es_idx], y_tr[es_idx])],
            callbacks=[lgb.early_stopping(100, verbose=False)])
    p_gbm = sym_predict(gbm, val_df, p_val)
    results.append(dict(method="LightGBM stack", val_log_loss=round(log_loss(y_val, p_gbm, labels=[0, 1, 2]), 4)))
    models["LightGBM stack"] = gbm
    print("LightGBM best iteration:", gbm.best_iteration_)
except Exception as e:
    print("LightGBM skip ->", repr(e)[:200])

res = pd.DataFrame(results).sort_values("val_log_loss").reset_index(drop=True)
res["gain_vs_base"] = (base_ll - res["val_log_loss"]).round(4)
res.to_csv("stack_results.csv", index=False)
display(res)

best = res.iloc[0]["method"]
USE = best if (best != "base TTA" and base_ll - res.iloc[0]["val_log_loss"] >= 0.001) else "base TTA"
print("USING:", USE)

In [ ]:
if "LightGBM stack" in models:
    imp = pd.Series(models["LightGBM stack"].feature_importances_, index=X_tr.columns).sort_values(ascending=False)
    print("Top features (LightGBM):")
    print(imp.head(12).to_string())

## Submission

In [ ]:
probs = p_test if USE == "base TTA" else sym_predict(models[USE], test, p_test)
sub = pd.DataFrame({"id": test["id"], "winner_model_a": probs[:, 0],
                    "winner_model_b": probs[:, 1], "winner_tie": probs[:, 2]})
sub.to_csv("submission.csv", index=False)
print(sub.shape, "| method:", USE)
sub.head()